# Fixed-Chain FHMM — Variational EM

A **Factorial Hidden Markov Model** (FHMM) decomposes an observed time series $y(t)$ into
contributions from $K$ independent Markov chains running in parallel:

$$y(t) = \sum_{k=1}^{K} \mu_k[x_k(t)] + \varepsilon(t)$$

Each chain $x_k$ follows its own Markov dynamics with transition matrix $A_k$ and initial
distribution $\pi_k$. The emission mean $\mu_k[s]$ and variance $\sigma^2_k[s]$ are
state-specific, and the noise $\varepsilon(t) \sim \mathcal{N}(0, \sum_k \sigma^2_k[x_k(t)])$.

Because exact posterior inference couples all chains through the shared observation, `FHMMVariational`
uses **mean-field variational EM**:

- **E-step** — for each chain $c$, run forward-backward on the effective residual
  $y(t) - \sum_{d \neq c} \mathbb{E}_q[\mu_d(x_d(t))]$, returning marginals $\gamma_c(t,s)$
  and $\xi_c(t,i,j)$.
- **M-step** — update $\mu_c[s]$, $\sigma^2_c[s]$, $A_c$, and $\pi_c$ in closed form
  from those marginals.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from FHMMVariational import FHMMVariational

## 1. Generating Synthetic Data

We create three two-state chains with sticky transitions — each chain tends to stay in its current
state, switching with probability 0.1. The per-chain emission signals are summed with Gaussian noise
to produce the single mixed observation $y(t)$ that the model will see.

In [ ]:
np.random.seed(10012025)

n_chains, n_states, T = 3, 2, 200
fhmm = FHMMVariational(n_chains=n_chains, n_states=n_states)

hidden = np.zeros((T, n_chains), dtype=int)   # true hidden states
chain_signals = np.zeros((T, n_chains))        # noiseless per-chain emissions
obs = np.zeros(T)                              # mixed observation

for c in range(n_chains):
    z = np.zeros(T, dtype=int)
    z[0] = np.random.choice(n_states, p=[0.05, 0.95])
    for tt in range(1, T):
        p = [0.9, 0.1] if z[tt - 1] == 0 else [0.1, 0.9]
        z[tt] = np.random.choice(n_states, p=p)
    hidden[:, c] = z
    chain_signals[:, c] = fhmm.means[c][z]
    obs += fhmm.means[c][z] + np.random.normal(0, np.sqrt(fhmm.vars[c][z]))

t = np.arange(T)
print('Emission means used for data generation:')
for c in range(n_chains):
    print(f'  Chain {c}:  mu={fhmm.means[c].round(3)},  sigma2={fhmm.vars[c].round(3)}')

## 2. Individual Chain Signals

Each chain $k$ emits $\mu_k[x_k(t)]$ at every time step — a step function that alternates between
two levels as the hidden state switches. The shaded regions mark periods where the chain is in
state 1 (high-emission). These latent signals are what inference will attempt to recover.

In [ ]:
COLORS = ['#2196F3', '#4CAF50', '#FF5722']

fig, axes = plt.subplots(n_chains, 1, figsize=(12, 5), sharex=True)
for c, ax in enumerate(axes):
    ax.fill_between(t, 0, 1, where=(hidden[:, c] == 1),
                    transform=ax.get_xaxis_transform(),
                    color=COLORS[c], alpha=0.18)
    ax.plot(t, chain_signals[:, c], color=COLORS[c], lw=1.5)
    ax.set_ylabel(f'Chain {c}', fontsize=10)
    lo, hi = chain_signals.min() - 0.1, chain_signals.max() + 0.1
    ax.set_ylim(lo, hi)

axes[0].set_title(r'True individual chain signals  $\mu_k[x_k(t)]$', fontsize=12)
axes[-1].set_xlabel('Time step $t$')
plt.tight_layout()
plt.show()

## 3. The Mixed Observation

The model receives only $y(t)$ — the noisy superposition of all chain emissions. The individual
chain structures are completely hidden; recovering them from this single trace is the inference task.

In [ ]:
noiseless = chain_signals.sum(axis=1)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, obs, color='#555', lw=1, alpha=0.7, label='Observed $y(t)$')
ax.plot(t, noiseless, color='#E91E63', lw=1.5, ls='--',
        label=r'Noiseless sum  $\sum_k \mu_k[x_k(t)]$')
ax.set_xlabel('Time step $t$')
ax.set_ylabel('Amplitude')
ax.set_title(r'Mixed observation  $y(t) = \sum_k \mu_k[x_k(t)] + \varepsilon(t)$')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 4. Running Variational EM

We call `variational_inference`, which runs the E-step / M-step loop for `n_iter` iterations.
The M-step updates the emission means and variances, transition matrices, and initial distributions
for all three chains simultaneously, using residuals computed from the pre-update parameters.

In [ ]:
posterior, viterbi_paths = fhmm.variational_inference(obs, n_iter=500)

print('Learned emission means (sorted):')
for c in range(n_chains):
    print(f'  Chain {c}: {np.sort(fhmm.means[c]).round(4)}')
print('\nLearned transition matrices:')
for c in range(n_chains):
    print(f'  Chain {c}: stay-0={fhmm.A[c][0,0]:.3f}  stay-1={fhmm.A[c][1,1]:.3f}')

## 5. Posterior Marginals and Viterbi Paths

`variational_inference` returns for each chain:

- **Posterior** $\gamma_c(t, 1) = P(x_c(t) = 1 \mid y)$ — the soft probability of being in the
  high-emission state, shown as a continuous curve.
- **Viterbi path** — the most-likely state sequence, decoded by dynamic programming.

The true state-1 regions (shaded) provide a ground-truth reference.

In [ ]:
fig, axes = plt.subplots(n_chains, 1, figsize=(12, 6), sharex=True)
for c, ax in enumerate(axes):
    ax.fill_between(t, 0, 1, where=(hidden[:, c] == 1),
                    transform=ax.get_xaxis_transform(),
                    color=COLORS[c], alpha=0.12, label='True state 1')
    ax.plot(t, posterior[c][:, 1], color=COLORS[c], lw=1.5, alpha=0.9,
            label='Posterior $P(x=1 \\mid y)$')
    ax.step(t, viterbi_paths[c], color='k', lw=1.0, where='post',
            ls='--', alpha=0.6, label='Viterbi')
    ax.set_ylim(-0.08, 1.08)
    ax.set_ylabel(f'Chain {c}', fontsize=10)
    if c == 0:
        ax.legend(loc='upper right', ncol=3, fontsize=8)

axes[0].set_title('Posterior marginals and Viterbi paths per chain', fontsize=12)
axes[-1].set_xlabel('Time step $t$')
plt.tight_layout()
plt.show()

## 6. Signal Decomposition and Reconstruction

Using the posterior marginals we compute the **posterior-mean contribution** of each chain:

$$\hat{s}_c(t) = \sum_s \gamma_c(t, s)\, \mu_c[s]$$

Summing across chains gives the total reconstruction
$\hat{y}(t) = \sum_c \hat{s}_c(t)$,
which we compare against the original noisy observation.

In [ ]:
# Posterior-mean contribution of each chain: (T, n_chains)
recovered = np.column_stack([posterior[c] @ fhmm.means[c] for c in range(n_chains)])
reconstruction = recovered.sum(axis=1)

fig, axes = plt.subplots(n_chains + 1, 1, figsize=(12, 8), sharex=True)

for c, ax in enumerate(axes[:n_chains]):
    ax.plot(t, chain_signals[:, c], color='#bbb', lw=2.5, label='True signal')
    ax.plot(t, recovered[:, c], color=COLORS[c], lw=1.5, ls='--',
            label=r'Recovered $\hat{s}_c(t)$')
    ax.set_ylabel(f'Chain {c}', fontsize=10)
    if c == 0:
        ax.legend(loc='upper right', ncol=2, fontsize=8)

ax_tot = axes[-1]
ax_tot.plot(t, obs, color='#aaa', lw=1.0, alpha=0.8, label='Observed $y(t)$')
ax_tot.plot(t, reconstruction, color='#E91E63', lw=1.5, ls='--',
            label=r'Reconstruction $\hat{y}(t)$')
ax_tot.set_ylabel('Total', fontsize=10)
ax_tot.legend(loc='upper right', ncol=2, fontsize=8)
ax_tot.set_xlabel('Time step $t$')

axes[0].set_title('True vs recovered chain signals and total reconstruction', fontsize=12)
plt.tight_layout()
plt.show()